# ***04 · Sensibilidad del método para extraer parámetros***

**Objetivo:** punto 7 del plan. Se evalúa qué tan bien el método Monte Carlo recupera $\sin^2\theta_W$:

1. generar un "experimento" con un valor verdadero ($\sin^2\theta_W=0.23126$)
2. medir $A_{FB}$ con los eventos
3. invertir la curva teórica $A_{FB}(\sin^2\theta_W)$ para estimar $\sin^2\theta_W$
4. repetir muchas veces y cuantificar la dispersión

## ***a) El error de $A_{FB}$ decrece como $1/\sqrt{N}$***

In [ ]:
import sys
sys.path.append("../src")          # permite importar el paquete fbasym sin instalarlo

import numpy as np
import matplotlib.pyplot as plt
from fbasym import *
from fbasym import plots

n_rep = 100
tamanos = [1000, 10000, 100000]
A_p, B_p = coeficientes(M_Z)

In [ ]:
np.random.seed(1)
medidas_afb, dispersion = {}, []
for N in tamanos:
    medidas = np.array([asimetria_fb(rejection_sampling(A_p, B_p, N)[0]) for _ in range(n_rep)])
    medidas_afb[N] = medidas
    dispersion.append(np.std(medidas))
    print(f"N = {N:7d}   A_FB media = {np.mean(medidas):+.4f}   desviación = {np.std(medidas):.4f}   1/sqrt(N) = {1/np.sqrt(N):.4f}")

fig = plots.figura_error_vs_N(tamanos, dispersion, medidas_afb)
fig.savefig("../figures/04a_error_vs_N.png", dpi=150)
plt.show()

## ***b) Energía óptima para medir $\sin^2\theta_W$***

Lo relevante es cuánto cambia $A_{FB}$ al cambiar $\sin^2\theta_W$:

$$\sigma_{\sin^2\theta_W}\approx\frac{\sigma_{A_{FB}}}{\left|\partial A_{FB}/\partial\sin^2\theta_W\right|}$$

La derivada se calcula con diferencia finita (`derivada_afb`).

In [ ]:
for E in [60, 85, M_Z, 95, 105, 150]:
    d = derivada_afb(E)
    print(f"raiz_s = {E:8.4f} GeV   dA_FB/d(sin2w) = {d:+8.3f}   --> error en sin2w con 1/sqrt(N)=0.003: {0.003/abs(d):.4f}")

E_sens = np.linspace(50, 140, 600)
deriv_E = np.array([derivada_afb(E) for E in E_sens])
afb_sens = np.array([afb_teorica(E) for E in E_sens])
sigma_s2w = np.sqrt((1 - afb_sens**2) / 100000) / np.abs(deriv_E)

fig = plots.figura_sensibilidad_energia(E_sens, deriv_E, sigma_s2w)
fig.savefig("../figures/04b_sensibilidad_vs_energia.png", dpi=150)
plt.show()

La sensibilidad es máxima cerca del polo (la derivada vale $\approx-2.9$ a 85 GeV y $\approx-1.7$ en $m_Z$). Aunque $A_{FB}$ en $m_Z$ es pequeña, depende fuertemente de $\sin^2\theta_W$ porque $A_{FB}\propto(1-4\sin^2\theta_W)^2$ y ese factor casi se anula. Por encima de la resonancia (95–105 GeV) $A_{FB}$ es grande pero casi independiente de $\sin^2\theta_W$, y con el mismo número de eventos el error resulta 4 o 5 veces mayor.

## ***c) Extracción de $\sin^2\theta_W$ en el polo***

La curva $A_{FB}(\sin^2\theta_W)$ se invierte por interpolación en un intervalo donde es monótona ($0.20$ a $0.245$).

In [ ]:
extraer_s2w = construir_inversor_s2w(M_Z)

np.random.seed(7)
resultados = {}
for N in tamanos:
    est = [extraer_s2w(asimetria_fb(rejection_sampling(A_p, B_p, N)[0])) for _ in range(n_rep)]
    resultados[N] = np.array(est)
    print(f"N = {N:7d}   sin2w = {np.mean(est):.4f} +- {np.std(est):.4f}   (valor verdadero {S2W})")

sig_sim = [np.std(resultados[N]) for N in tamanos]
N_fino = np.logspace(3, 5, 50)
sig_teo = [error_afb(afb_teorica(M_Z), N) / abs(derivada_afb(M_Z)) for N in N_fino]

fig = plots.figura_extraccion(tamanos, resultados, sig_sim, N_fino, sig_teo, S2W)
fig.savefig("../figures/04c_extraccion_sin2w.png", dpi=150)
plt.show()